<a href="https://colab.research.google.com/github/NomadZhang/DSA5204/blob/main/03_training_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!git clone https://github.com/NomadZhang/DSA5204.git

%cd DSA5204

!pip install torch transformers datasets accelerate

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import load_dataset
from src.common import get_device
from src.lora import inject_lora

device = get_device()
print(f"Current cloud-based devices: {device}")

model_id = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

# Load the model and tokenizer
print("Model and tokenizer is being loaded...")
tokenizer = AutoTokenizer.from_pretrained(model_id)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype=torch.float16)

for param in model.parameters():
    param.requires_grad = False
model = inject_lora(model, target_modules=("q_proj", "v_proj"), r=8, alpha=16)

model.to(device)
print("Lora injected! Everything is ready!")

In [ ]:
# Load the dataset
dataset = load_dataset('json', data_files='./data/train.jsonl')

def tokenize_function(examples):
    # Transform text into Token IDs
    texts = [p + r for p, r in zip(examples['prompt'], examples['response'])]
    return tokenizer(texts, padding="max_length", truncation=True, max_length=256)

print("Dataset is being processed...")
tokenized_datasets = dataset.map(tokenize_function, batched=True)
print("Dataset has been processed!")

In [ ]:
from transformers import Trainer, TrainingArguments, DataCollatorForLanguageModeling
import torch

for param in model.parameters():
    if param.requires_grad:
        param.data = param.data.to(torch.float32)

# Tell model how to learn
training_args = TrainingArguments(
    output_dir="./lora_results",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    num_train_epochs=1,
    logging_steps=10,
    save_steps=100,
    fp16=True,
)

# Auto-calculate the loss
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    data_collator=data_collator,
)

print("🚀 Light the fire! Start the training...")
trainer.train()

# Save *only* the LoRA adapter weights and config
model.save_pretrained("./tinyllama-lora-finetuned")
print("🎉 Training completed!")